# 強化学習 実践入門

**多腕バンディットから Q-learning まで、実装して理解するハンズオン教材**

この Notebook では、強化学習（Reinforcement Learning; RL）の基本概念を、数式・実装・可視化を往復しながら学びます。外部の RL ライブラリは使わず、`NumPy` と `Matplotlib` だけでアルゴリズムの中身を実装します。

## 学習目標

学習後には、次のことができるようになります。

1. 状態・行動・報酬・方策・価値関数を説明できる
2. 探索と活用のトレードオフを実験で確認できる
3. ε-greedy 法と Q-learning をゼロから実装できる
4. 学習曲線と獲得方策を読み、ハイパーパラメータを調整できる

## 進め方

- 上から順にセルを実行してください。
- 実行時間の目安は 15〜30 分です。
- `TODO` の演習に取り組んでから、折りたたみ可能な解答例を確認してください。

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

SEED = 42
np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")

Python: 3.12.13
NumPy: 2.4.6


## 1. 強化学習の全体像

強化学習では、**エージェント**が**環境**と繰り返し相互作用します。

1. 環境から状態 $S_t$ を観測する
2. 方策 $\pi$ に従って行動 $A_t$ を選ぶ
3. 環境から報酬 $R_{t+1}$ と次状態 $S_{t+1}$ を受け取る
4. 将来を含む報酬の合計が大きくなるように学習する

割引収益（return）は次式です。

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots$$

$\gamma \in [0, 1]$ は割引率です。小さいほど目先の報酬を、大きいほど将来の報酬を重視します。

### 教師あり学習との違い

| 観点 | 教師あり学習 | 強化学習 |
|---|---|---|
| 教師信号 | 正解ラベル | 報酬 |
| データ | 固定データセットが中心 | 行動によってデータが変化 |
| 目的 | 予測誤差の最小化 | 長期的な累積報酬の最大化 |
| 難しさ | 汎化 | 探索、遅延報酬、信用割当 |

### ウォームアップ：割引収益を計算する

報酬列 $[1, 0, 2, 3]$ の各時点から見た割引収益を、後ろから再帰的に計算します。

$$G_t = R_{t+1} + \gamma G_{t+1}$$

In [ ]:
def discounted_returns(rewards, gamma=0.9):
    """各時点からの割引収益を返す。"""
    returns = np.zeros(len(rewards), dtype=float)
    running_return = 0.0
    for t in reversed(range(len(rewards))):
        running_return = rewards[t] + gamma * running_return
        returns[t] = running_return
    return returns

rewards = [1, 0, 2, 3]
for gamma in [0.0, 0.5, 0.9, 1.0]:
    print(f"gamma={gamma:.1f}: {discounted_returns(rewards, gamma)}")

## 2. 多腕バンディット：探索と活用

多腕バンディットは「状態遷移のない」最小の強化学習問題です。各腕（行動）には未知の平均報酬があり、限られた試行で報酬を最大化します。

- **活用（exploitation）**：現在もっとも良いと推定した腕を選ぶ
- **探索（exploration）**：情報を得るため、別の腕も試す

ここでは ε-greedy 法を使います。確率 $1-\epsilon$ で最良の腕を、確率 $\epsilon$ でランダムな腕を選びます。

In [ ]:
@dataclass
class GaussianBandit:
    true_values: np.ndarray
    noise_std: float = 1.0

    def pull(self, action, rng):
        return rng.normal(self.true_values[action], self.noise_std)


def epsilon_greedy(q_values, epsilon, rng):
    """ε-greedy で行動を選ぶ。同率最大ならランダムに決める。"""
    if rng.random() < epsilon:
        return int(rng.integers(len(q_values)))
    best_actions = np.flatnonzero(q_values == q_values.max())
    return int(rng.choice(best_actions))


def run_bandit(true_values, epsilon=0.1, steps=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    env = GaussianBandit(np.asarray(true_values))
    q_values = np.zeros(len(true_values))
    counts = np.zeros(len(true_values), dtype=int)
    rewards = np.zeros(steps)
    optimal = np.zeros(steps)
    best_action = int(np.argmax(true_values))

    for t in range(steps):
        action = epsilon_greedy(q_values, epsilon, rng)
        reward = env.pull(action, rng)
        counts[action] += 1
        # 標本平均を逐次更新: Q <- Q + (R - Q) / N
        q_values[action] += (reward - q_values[action]) / counts[action]
        rewards[t] = reward
        optimal[t] = (action == best_action)

    return {"q_values": q_values, "counts": counts,
            "rewards": rewards, "optimal": optimal}


true_values = np.array([0.2, 1.0, 0.5, 1.5, -0.2])
result = run_bandit(true_values, epsilon=0.1)
print("真の平均報酬 :", true_values)
print("推定した価値 :", result["q_values"])
print("各腕の選択回数:", result["counts"])

### ε の違いを比較する

1 回の実験には偶然性があります。そこで複数の乱数 seed で実験し、平均累積報酬と最適行動率を比較します。

In [ ]:
def compare_epsilons(true_values, epsilons, steps=1000, runs=200):
    summary = {}
    for epsilon in epsilons:
        rewards = []
        optimal = []
        for seed in range(runs):
            out = run_bandit(true_values, epsilon, steps, seed)
            rewards.append(out["rewards"])
            optimal.append(out["optimal"])
        summary[epsilon] = {
            "reward": np.mean(rewards, axis=0),
            "optimal": np.mean(optimal, axis=0),
        }
    return summary


epsilons = [0.0, 0.01, 0.1, 0.3]
bandit_summary = compare_epsilons(true_values, epsilons)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for epsilon, data in bandit_summary.items():
    axes[0].plot(np.cumsum(data["reward"]), label=f"ε={epsilon}")
    axes[1].plot(np.convolve(data["optimal"], np.ones(50) / 50, mode="valid"),
                 label=f"ε={epsilon}")
axes[0].set(title="平均累積報酬", xlabel="ステップ", ylabel="累積報酬")
axes[1].set(title="最適行動率（50 step 移動平均）", xlabel="ステップ", ylabel="割合", ylim=(0, 1.05))
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

### 演習 1：非定常バンディット

現実では腕の価値が時間とともに変化することがあります。標本平均は過去の全履歴を同じ重みで扱うため、変化への追従が遅くなります。

次の更新式を、固定ステップサイズ $\alpha=0.1$ に変更してください。

$$Q(A_t) \leftarrow Q(A_t) + \alpha [R_t - Q(A_t)]$$

考えてみよう：標本平均と固定 $\alpha$ のどちらが、最近の報酬を強く反映するでしょうか。

In [ ]:
# TODO: 下の ??? を埋めてください（実行可能な初期値も入れてあります）
q_old, reward, alpha = 0.5, 1.2, 0.1
q_new = q_old + alpha * (reward - q_old)  # ???
print(f"更新前={q_old:.3f}, 更新後={q_new:.3f}")
assert np.isclose(q_new, 0.57), "更新式を確認しましょう"

<details><summary>演習 1 の解答とポイント</summary>

```python
q_new = q_old + alpha * (reward - q_old)
```

固定ステップサイズでは直近の観測ほど大きな重みを持つため、非定常な環境に追従しやすくなります。一方、定常環境では標本平均が安定した推定値になります。
</details>

## 3. マルコフ決定過程（MDP）

状態遷移を含む強化学習問題は、一般に MDP として表現します。

- 状態集合 $\mathcal{S}$
- 行動集合 $\mathcal{A}$
- 遷移確率 $P(s'\mid s,a)$
- 報酬 $R(s,a,s')$
- 割引率 $\gamma$

行動価値関数 $Q^\pi(s,a)$ は「状態 $s$ で行動 $a$ を選び、その後は方策 $\pi$ に従ったときの期待収益」です。

Q-learning は、最適行動価値を次の TD（Temporal Difference）更新で学習します。

$$Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\left[R_{t+1}+\gamma\max_a Q(S_{t+1},a)-Q(S_t,A_t)\right]$$

角括弧の中を **TD 誤差** と呼びます。終端状態では将来価値を 0 とします。

## 4. GridWorld を作る

5×5 の盤面で、エージェントをスタート `S` からゴール `G` へ導きます。

- 行動：上・右・下・左
- 通常の移動：$-1$（短い経路を促す）
- 穴 `X`：$-10$ で終了
- ゴール `G`：$+10$ で終了
- 壁や盤外への移動：同じ場所に留まる

In [ ]:
class GridWorld:
    ACTIONS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
    ACTION_NAMES = np.array(["↑", "→", "↓", "←"])

    def __init__(self, rows=5, cols=5):
        self.rows, self.cols = rows, cols
        self.start = (0, 0)
        self.goal = (4, 4)
        self.holes = {(1, 3), (3, 1)}
        self.walls = {(1, 1), (2, 1), (3, 3)}
        self.n_states = rows * cols
        self.n_actions = len(self.ACTIONS)
        self.state = self.start

    def state_to_id(self, state):
        return state[0] * self.cols + state[1]

    def id_to_state(self, state_id):
        return divmod(state_id, self.cols)

    def reset(self):
        self.state = self.start
        return self.state_to_id(self.state)

    def step(self, action):
        candidate = tuple(np.array(self.state) + self.ACTIONS[action])
        valid = (0 <= candidate[0] < self.rows and
                 0 <= candidate[1] < self.cols and
                 candidate not in self.walls)
        if valid:
            self.state = candidate

        terminated = self.state == self.goal or self.state in self.holes
        reward = 10.0 if self.state == self.goal else (-10.0 if self.state in self.holes else -1.0)
        return self.state_to_id(self.state), reward, terminated

    def render(self, path=None):
        path = set(path or [])
        for r in range(self.rows):
            row = []
            for c in range(self.cols):
                pos = (r, c)
                symbol = ("S" if pos == self.start else "G" if pos == self.goal else
                          "X" if pos in self.holes else "#" if pos in self.walls else
                          "·" if pos in path else ".")
                row.append(symbol)
            print(" ".join(row))


env = GridWorld()
env.render()

### ランダム方策を試す

学習前のエージェントがどの程度うまくいくか、基準値を確認します。安全のため 1 エピソードの最大ステップ数を設けます。

In [ ]:
def run_random_policy(env, episodes=1000, max_steps=100, seed=SEED):
    rng = np.random.default_rng(seed)
    returns, successes = [], []
    for _ in range(episodes):
        env.reset()
        total_reward = 0.0
        success = False
        for _ in range(max_steps):
            action = int(rng.integers(env.n_actions))
            _, reward, terminated = env.step(action)
            total_reward += reward
            if terminated:
                success = env.state == env.goal
                break
        returns.append(total_reward)
        successes.append(success)
    return np.asarray(returns), np.asarray(successes)

random_returns, random_successes = run_random_policy(env)
print(f"ランダム方策の平均収益: {random_returns.mean():.2f}")
print(f"ランダム方策の成功率  : {random_successes.mean():.1%}")

## 5. Q-learning を実装する

学習中は ε-greedy で探索し、評価時は最大の Q 値を持つ行動を選びます。ε はエピソードごとに減衰させ、序盤は広く探索、終盤は学んだ知識を活用します。

In [ ]:
def train_q_learning(env, episodes=1000, alpha=0.1, gamma=0.95,
                     epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995,
                     max_steps=100, seed=SEED):
    rng = np.random.default_rng(seed)
    q_table = np.zeros((env.n_states, env.n_actions))
    episode_returns = np.zeros(episodes)
    episode_lengths = np.zeros(episodes, dtype=int)
    epsilon_history = np.zeros(episodes)

    for episode in range(episodes):
        state = env.reset()
        total_reward = 0.0
        epsilon_history[episode] = epsilon

        for step in range(max_steps):
            action = epsilon_greedy(q_table[state], epsilon, rng)
            next_state, reward, terminated = env.step(action)

            # 終端なら将来価値は 0
            next_value = 0.0 if terminated else np.max(q_table[next_state])
            td_target = reward + gamma * next_value
            td_error = td_target - q_table[state, action]
            q_table[state, action] += alpha * td_error

            state = next_state
            total_reward += reward
            if terminated:
                break

        episode_returns[episode] = total_reward
        episode_lengths[episode] = step + 1
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    history = {"returns": episode_returns, "lengths": episode_lengths,
               "epsilons": epsilon_history}
    return q_table, history


q_table, history = train_q_learning(env)
print("Q-table shape:", q_table.shape)
print("開始状態の Q 値:", q_table[env.state_to_id(env.start)])

In [ ]:
def moving_average(values, window=50):
    if len(values) < window:
        return np.asarray(values)
    return np.convolve(values, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["returns"], alpha=0.2, color="tab:blue")
axes[0].plot(np.arange(49, len(history["returns"])), moving_average(history["returns"]), color="tab:blue")
axes[0].set(title="エピソード収益", xlabel="エピソード", ylabel="収益")
axes[1].plot(np.arange(49, len(history["lengths"])), moving_average(history["lengths"]), color="tab:orange")
axes[1].set(title="エピソード長（移動平均）", xlabel="エピソード", ylabel="ステップ数")
axes[2].plot(history["epsilons"], color="tab:green")
axes[2].set(title="探索率 ε", xlabel="エピソード", ylabel="ε")
plt.tight_layout()
plt.show()

### 学習した方策を可視化する

各マスで最大の Q 値を持つ行動を矢印で表示します。未訪問状態では Q 値がすべて同じなので `?` と表示します。

In [ ]:
def show_policy(env, q_table):
    for r in range(env.rows):
        row = []
        for c in range(env.cols):
            pos = (r, c)
            if pos == env.goal:
                symbol = "G"
            elif pos in env.holes:
                symbol = "X"
            elif pos in env.walls:
                symbol = "#"
            else:
                values = q_table[env.state_to_id(pos)]
                symbol = "?" if np.allclose(values, values[0]) else env.ACTION_NAMES[np.argmax(values)]
            row.append(symbol)
        print(" ".join(row))


show_policy(env, q_table)

In [ ]:
def evaluate_greedy_policy(env, q_table, episodes=100, max_steps=100):
    returns, successes, final_path = [], [], []
    for _ in range(episodes):
        state = env.reset()
        path = [env.state]
        total_reward = 0.0
        for _ in range(max_steps):
            action = int(np.argmax(q_table[state]))
            state, reward, terminated = env.step(action)
            path.append(env.state)
            total_reward += reward
            if terminated:
                break
        returns.append(total_reward)
        successes.append(env.state == env.goal)
        final_path = path
    return np.asarray(returns), np.asarray(successes), final_path


eval_returns, eval_successes, learned_path = evaluate_greedy_policy(env, q_table)
print(f"学習済み方策の平均収益: {eval_returns.mean():.2f}")
print(f"学習済み方策の成功率  : {eval_successes.mean():.1%}")
print("\n獲得した経路（·）:")
env.render(learned_path)

assert eval_successes.mean() >= 0.95, "学習が不安定です。エピソード数や乱数 seed を見直してください。"

### 演習 2：TD 更新を手計算する

次の条件で更新後の $Q(s,a)$ を計算してください。

- 現在値：$Q(s,a)=2.0$
- 報酬：$r=1.0$
- 次状態の最大行動価値：$\max_{a'}Q(s',a')=4.0$
- 学習率：$\alpha=0.1$
- 割引率：$\gamma=0.9$

In [ ]:
# TODO: 式を確認し、値を変更して実験してみましょう
q, reward, next_max_q, alpha, gamma = 2.0, 1.0, 4.0, 0.1, 0.9
td_target = reward + gamma * next_max_q
td_error = td_target - q
updated_q = q + alpha * td_error

print(f"TD target = {td_target:.2f}")
print(f"TD error  = {td_error:.2f}")
print(f"更新後 Q  = {updated_q:.2f}")
assert np.isclose(updated_q, 2.26)

<details><summary>演習 2 の解答</summary>

$$\text{TD target}=1.0+0.9\times4.0=4.6$$
$$\text{TD error}=4.6-2.0=2.6$$
$$Q_{new}=2.0+0.1\times2.6=2.26$$

正の TD 誤差なので、その行動の価値推定は上昇します。
</details>

### 演習 3：ハイパーパラメータ実験

学習率 $\alpha$ を変え、最後の 100 エピソードの平均収益を比較します。

観察のヒント：

- $\alpha$ が小さすぎると学習に時間がかかる
- $\alpha$ が大きすぎると推定値が振動しやすい
- 結果は環境・報酬の確率性・学習期間にも依存する

In [ ]:
# TODO: 候補に 0.01 や 1.0 を追加して比較してみましょう
alphas = [0.05, 0.1, 0.5]
alpha_results = {}

for alpha_value in alphas:
    _, h = train_q_learning(GridWorld(), alpha=alpha_value, seed=SEED)
    alpha_results[alpha_value] = h["returns"]
    print(f"alpha={alpha_value:>4}: 最後の100話の平均収益 = {h['returns'][-100:].mean():.2f}")

plt.figure(figsize=(9, 4))
for alpha_value, returns in alpha_results.items():
    plt.plot(np.arange(49, len(returns)), moving_average(returns), label=f"α={alpha_value}")
plt.xlabel("エピソード")
plt.ylabel("収益（50話移動平均）")
plt.title("学習率による学習曲線の違い")
plt.legend()
plt.show()

## 6. 発展課題

理解を深めるには、次の順で改造してみてください。

1. **報酬設計**：通常報酬を `-0.1`、ゴールを `+1` に変え、方策が変わるか確認する
2. **確率的遷移**：10% の確率で意図と異なる方向へ動くよう `step` を変更する
3. **ε のスケジュール**：線形減衰や指数減衰を比較する
4. **SARSA**：次状態で実際に選んだ行動 $A_{t+1}$ を使う on-policy 更新を実装する
5. **Double Q-learning**：Q 値の過大評価を抑える方法を調べ、実装する

SARSA の更新式：

$$Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha[R_{t+1}+\gamma Q(S_{t+1},A_{t+1})-Q(S_t,A_t)]$$

Q-learning は学習時の行動方策と評価対象の greedy 方策が異なる **off-policy**、SARSA は実際の行動を評価する **on-policy** です。

## 7. 実務でのチェックリスト

- **MDP として妥当か**：状態に意思決定に必要な情報が含まれるか
- **報酬ハッキングはないか**：代理指標だけを攻略する抜け道がないか
- **ベースラインはあるか**：ランダム方策・ルールベース手法と比較したか
- **複数 seed で評価したか**：平均だけでなく分散や信頼区間も見たか
- **学習と評価を分離したか**：評価時には探索を切ったか
- **安全制約があるか**：実環境で危険な探索を許していないか
- **オフライン評価を検討したか**：シミュレータや過去ログを活用できないか

強化学習は「予測」ではなく「介入」を学びます。そのため、オンライン広告・推薦・ロボットなどでは、性能だけでなく安全性、公平性、説明可能性、実験コストも重要です。

## 8. まとめ

- 強化学習は、環境との相互作用から累積報酬を最大化する方策を学ぶ
- ε-greedy は探索と活用を単純かつ有効に両立する
- Q-learning は TD 誤差を使って、モデルなしで最適行動価値を学ぶ
- 学習曲線、複数 seed、ベースラインによる評価が重要
- 実務では報酬設計と安全な探索が成否を左右する

次の一歩として、Gymnasium の `FrozenLake` や `CartPole` を試し、状態が大きい場合に Q-table をニューラルネットワークへ置き換える DQN へ進むと理解がつながります。

## 理解度チェック

1. ε を 0 にすると、どのような問題が起こり得ますか？
2. 終端状態の次状態価値を 0 とするのはなぜですか？
3. 学習率 $\alpha$ と割引率 $\gamma$ は、それぞれ何を制御しますか？
4. Q-learning が off-policy と呼ばれる理由を説明してください。
5. 学習済み方策を 1 回だけでなく複数回評価すべき理由は何ですか？

<details><summary>解答例</summary>

1. 初期に得た不正確な推定へ固定され、より良い行動を発見できない可能性があります。
2. エピソード終了後には将来の報酬が存在しないためです。
3. $\alpha$ は新しい情報をどれだけ強く反映するか、$\gamma$ は将来報酬をどれだけ重視するかを制御します。
4. 行動は ε-greedy で選びながら、更新対象には greedy な最大 Q 値を使うためです。
5. 環境や方策に確率性がある場合、1 回の結果は偶然に左右されるためです。
</details>